# ECNN V4: End-to-End Equivariant Bottleneck

This notebook defines and tests an ECNN V4 architecture that keeps equivariance through the bottleneck path for reconstruction.

## Design goals
- Keep C4 equivariant feature fields through encoder, bottleneck, and decoder
- Remove group-pooling bottleneck from the reconstruction path
- Remove dense fc bottleneck and repeat-based lifting
- Add pure equivariance sanity checks before anomaly scoring

In [ ]:
# Environment setup
import os
import sys
import random
import subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

try:
    from e2cnn import gspaces
    from e2cnn import nn as e2nn
except Exception:
    print('Installing e2cnn...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'e2cnn'])
    from e2cnn import gspaces
    from e2cnn import nn as e2nn

print('e2cnn ready.')

In [ ]:
# Model definition: ECNNAutoencoderV4
class ECNNAutoencoderV4(nn.Module):
    """
    V4 ECNN with an equivariant bottleneck path for reconstruction.

    Key differences from V3-style bottlenecks:
    - No group pooling before reconstruction bottleneck
    - No fc encode/decode latent bridge
    - No channel repeat lifting
    """

    def __init__(self):
        super().__init__()

        self.r2_act = gspaces.Rot2dOnR2(N=4)
        self.in_type = e2nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])

        self.type_128 = e2nn.FieldType(self.r2_act, 32 * [self.r2_act.regular_repr])
        self.type_256 = e2nn.FieldType(self.r2_act, 64 * [self.r2_act.regular_repr])
        self.type_512 = e2nn.FieldType(self.r2_act, 128 * [self.r2_act.regular_repr])
        self.type_1024 = e2nn.FieldType(self.r2_act, 256 * [self.r2_act.regular_repr])

        # 128 -> 64 -> 32 -> 16 -> 8
        self.enc1 = self._down_block(self.in_type, self.type_128, k=7, s=2, p=3)
        self.enc2 = self._down_block(self.type_128, self.type_256, k=3, s=2, p=1)
        self.enc3 = self._down_block(self.type_256, self.type_512, k=3, s=2, p=1)
        self.enc4 = self._down_block(self.type_512, self.type_1024, k=3, s=2, p=1)

        # Equivariant bottleneck at 8x8
        self.bottleneck = nn.Sequential(
            e2nn.R2Conv(self.type_1024, self.type_1024, kernel_size=3, padding=1, bias=False),
            e2nn.InnerBatchNorm(self.type_1024),
            e2nn.ReLU(self.type_1024),
            e2nn.R2Conv(self.type_1024, self.type_1024, kernel_size=3, padding=1, bias=False),
            e2nn.InnerBatchNorm(self.type_1024),
            e2nn.ReLU(self.type_1024),
        )

        # 8 -> 16 -> 32 -> 64 -> 128
        self.up1 = self._up_block(self.type_1024, self.type_512)
        self.up2 = self._up_block(self.type_512, self.type_256)
        self.up3 = self._up_block(self.type_256, self.type_128)
        self.final_conv = e2nn.R2Conv(self.type_128, self.in_type, kernel_size=3, padding=1)

        self.sigmoid = nn.Sigmoid()

    def _down_block(self, in_type, out_type, k=3, s=2, p=1):
        return nn.Sequential(
            e2nn.R2Conv(in_type, out_type, kernel_size=k, stride=s, padding=p, bias=False),
            e2nn.InnerBatchNorm(out_type),
            e2nn.ReLU(out_type),
        )

    def _up_block(self, in_type, out_type):
        return nn.Sequential(
            e2nn.R2Conv(in_type, out_type, kernel_size=3, padding=1, bias=False),
            e2nn.InnerBatchNorm(out_type),
            e2nn.ReLU(out_type),
        )

    def _upsample_geo(self, x_geo, out_type):
        x_up = F.interpolate(x_geo.tensor, scale_factor=2, mode='nearest')
        return e2nn.GeometricTensor(x_up, out_type)

    def forward(self, x):
        x_geo = e2nn.GeometricTensor(x, self.in_type)

        e1 = self.enc1(x_geo)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        b = self.bottleneck(e4)

        d1 = self._upsample_geo(b, self.type_1024)
        d1 = self.up1(d1)

        d2 = self._upsample_geo(d1, self.type_512)
        d2 = self.up2(d2)

        d3 = self._upsample_geo(d2, self.type_256)
        d3 = self.up3(d3)

        d4 = self._upsample_geo(d3, self.type_128)
        out = self.final_conv(d4)

        return self.sigmoid(out.tensor)

    @torch.no_grad()
    def compute_reconstruction_error(self, x, mode='squared'):
        recon = self.forward(x)
        if mode == 'abs':
            return torch.abs(x - recon)
        if mode == 'squared':
            return (x - recon) ** 2
        raise ValueError(f'Unknown mode: {mode}')

In [ ]:
# Instantiate and sanity-check shapes
model_v4 = ECNNAutoencoderV4().to(DEVICE)
model_v4.eval()

x = torch.randn(2, 1, 128, 128, device=DEVICE)
with torch.no_grad():
    y = model_v4(x)

print('Input shape :', tuple(x.shape))
print('Output shape:', tuple(y.shape))

In [ ]:
# Pure equivariance sanity metric (pre-scoring, pre-thresholding)
@torch.no_grad()
def rotate_tensor(x, k):
    return torch.rot90(x, k=k, dims=(-2, -1))

@torch.no_grad()
def pure_equivariance_gap(model, batch_size=8, n_batches=8):
    model.eval()
    eps = 1e-8

    recon_abs = 0.0
    recon_norm = 0.0
    err_abs = 0.0
    err_norm = 0.0
    n = 0

    for _ in range(n_batches):
        x = torch.rand(batch_size, 1, 128, 128, device=DEVICE)
        f0 = model(x)
        e0 = (x - f0).pow(2)

        for k in [1, 2, 3]:
            xr = rotate_tensor(x, k)
            fr = model(xr)
            er = (xr - fr).pow(2)

            f0r = rotate_tensor(f0, k)
            e0r = rotate_tensor(e0, k)

            rg = torch.mean(torch.abs(fr - f0r), dim=[1, 2, 3])
            rb = torch.mean(torch.abs(f0r), dim=[1, 2, 3])

            eg = torch.mean(torch.abs(er - e0r), dim=[1, 2, 3])
            eb = torch.mean(torch.abs(e0r), dim=[1, 2, 3])

            recon_abs += float(rg.sum().item())
            recon_norm += float((rg / (rb + eps)).sum().item())
            err_abs += float(eg.sum().item())
            err_norm += float((eg / (eb + eps)).sum().item())
            n += int(x.size(0))

    return {
        'mean_recon_abs_gap': recon_abs / n,
        'mean_recon_norm_gap': recon_norm / n,
        'mean_error_abs_gap': err_abs / n,
        'mean_error_norm_gap': err_norm / n,
    }

metrics = pure_equivariance_gap(model_v4)
print('V4 pure equivariance metrics (lower is better):')
for k, v in metrics.items():
    print(f'  {k}: {v:.6f}')

## Optional Training Scaffold

Use this scaffold to train V4 with your existing data pipeline. This keeps notebook setup light and lets you adapt quickly to Colab/local paths.

In [ ]:
# Minimal dataset + train loop scaffold
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SliceDataset(Dataset):
    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        p = self.files[idx]
        if str(p).lower().endswith('.npy'):
            arr = np.load(p).astype(np.float32)
        else:
            arr = np.array(Image.open(p).convert('L')).astype(np.float32) / 255.0

        if arr.ndim == 3:
            arr = arr.squeeze()

        x = torch.from_numpy(arr).float().unsqueeze(0)
        if x.shape[-2:] != (128, 128):
            x = F.interpolate(x.unsqueeze(0), size=(128, 128), mode='bilinear', align_corners=False).squeeze(0)
        return x, x

def train_one_epoch(model, loader, optimizer):
    model.train()
    running = 0.0
    n = 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        recon = model(x)
        loss = F.mse_loss(recon, y)
        loss.backward()
        optimizer.step()

        b = int(x.size(0))
        running += float(loss.item()) * b
        n += b
    return running / max(1, n)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    running = 0.0
    n = 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        recon = model(x)
        loss = F.mse_loss(recon, y)
        b = int(x.size(0))
        running += float(loss.item()) * b
        n += b
    return running / max(1, n)

print('Training scaffold ready.')

In [ ]:
# Turbo-zip data setup (same style as notebook 8 ECNN optimized)
import shutil
import zipfile

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
    REPO_ROOT = REPO_ROOT.parent

IN_COLAB = Path('/content').exists()
PROJECT_ROOT = Path('/content/drive/MyDrive/symAD-ECNN') if IN_COLAB else REPO_ROOT
DATA_ROOT = PROJECT_ROOT / 'data'
LOCAL_DIR = Path('/content/local_data') if IN_COLAB else (REPO_ROOT / '.local_data')

ZIPS = {
    'train': DATA_ROOT / 'train_fast.zip',
    'val': DATA_ROOT / 'val_fast.zip',
}

def extract_zip(zip_path: Path, out_dir: Path, clean: bool = True):
    if clean and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)

def list_data_files(root: Path):
    if not root.exists():
        return []
    files = []
    for pat in ('*.npy', '*.png', '*.jpg', '*.jpeg'):
        files.extend(root.rglob(pat))
    return sorted([p for p in files if p.is_file()])

missing = [str(p) for p in ZIPS.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing turbo zip files. Expected: train_fast.zip and val_fast.zip. Missing: ' + ', '.join(missing)
    )

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_ROOT: {DATA_ROOT}')
print(f'LOCAL_DIR: {LOCAL_DIR}')
print('Extracting turbo zips...')
for split, zip_path in ZIPS.items():
    target_dir = LOCAL_DIR / split
    extract_zip(zip_path, target_dir, clean=True)
    print(f'  {split}: {zip_path} -> {target_dir}')

train_dir = LOCAL_DIR / 'train'
val_dir = LOCAL_DIR / 'val'

train_files = list_data_files(train_dir)
val_files = list_data_files(val_dir)

print(f'Train files: {len(train_files)}')
print(f'Val files  : {len(val_files)}')

if len(train_files) == 0:
    raise RuntimeError('No training files found after turbo extraction. Check zip contents.')

In [ ]:
# Train V4 (run only after confirming data paths)
if len(train_files) > 0:
    train_ds = SliceDataset(train_files)
    val_ds = SliceDataset(val_files) if len(val_files) > 0 else None

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0) if val_ds is not None else None

    model_v4 = ECNNAutoencoderV4().to(DEVICE)
    optimizer = torch.optim.Adam(model_v4.parameters(), lr=1e-4, weight_decay=1e-5)

    epochs = 100
    patience = 10
    no_improve = 0
    min_delta = 1e-6

    best_val = float('inf')
    save_dir = REPO_ROOT / 'models' / 'saved_models' / 'ecnn_v4'
    save_dir.mkdir(parents=True, exist_ok=True)
    best_path = save_dir / 'ecnn_v4_best.pth'

    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model_v4, train_loader, optimizer)
        if val_loader is not None:
            va_loss = validate(model_v4, val_loader)
        else:
            va_loss = tr_loss

        print(f'Epoch {epoch:03d} | train={tr_loss:.6f} | val={va_loss:.6f}')

        if va_loss < (best_val - min_delta):
            best_val = va_loss
            no_improve = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_v4.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val,
                'model_name': 'ECNNAutoencoderV4',
                'config': {'variant': 'v4_end_to_end_equivariant_bottleneck'}
            }, best_path)
            print(f'  Saved best: {best_path}')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch} (no val improvement for {patience} epochs).')
                break

    print(f'Best val loss: {best_val:.6f}')
    print(f'Best checkpoint: {best_path}')
else:
    print('Skipping training because no files were found.')

In [ ]:
# Evaluate V4 on turbo val/test splits (FPR@20 policy + reconstruction metrics)
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

TARGET_FPR = 0.20
EVAL_BATCH_SIZE = 16

test_zip = DATA_ROOT / 'test_fast.zip'
if not test_zip.exists():
    raise FileNotFoundError(f'Missing test turbo zip: {test_zip}')

test_dir = LOCAL_DIR / 'test'
extract_zip(test_zip, test_dir, clean=True)
test_files = list_data_files(test_dir)
print(f'Test files: {len(test_files)}')

if len(val_files) == 0 or len(test_files) == 0:
    raise RuntimeError('Need non-empty val and test files for evaluation.')

# Load best checkpoint (uses trained model if still in memory, otherwise restores from disk)
best_path = REPO_ROOT / 'models' / 'saved_models' / 'ecnn_v4' / 'ecnn_v4_best.pth'
if 'model_v4' in globals() and isinstance(model_v4, nn.Module):
    eval_model = model_v4.to(DEVICE).eval()
    print('Using in-memory model_v4 for evaluation.')
elif best_path.exists():
    eval_model = ECNNAutoencoderV4().to(DEVICE)
    ckpt = torch.load(best_path, map_location=DEVICE)
    state_dict = ckpt.get('model_state_dict', ckpt)
    eval_model.load_state_dict(state_dict, strict=False)
    eval_model.eval()
    print(f'Loaded checkpoint: {best_path}')
else:
    raise FileNotFoundError('No in-memory model and no saved checkpoint found for V4.')

val_ds_eval = SliceDataset(val_files)
test_ds_eval = SliceDataset(test_files)
val_loader_eval = DataLoader(val_ds_eval, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)
test_loader_eval = DataLoader(test_ds_eval, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

@torch.no_grad()
def compute_reconstruction_scores(model, loader):
    model.eval()
    mse_scores = []
    mae_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        recon = model(x)
        diff = x - recon
        mse = diff.pow(2).view(x.size(0), -1).mean(dim=1)
        mae = diff.abs().view(x.size(0), -1).mean(dim=1)
        mse_scores.extend(mse.detach().cpu().numpy().tolist())
        mae_scores.extend(mae.detach().cpu().numpy().tolist())
    return (
        np.asarray(mse_scores, dtype=np.float32),
        np.asarray(mae_scores, dtype=np.float32),
    )

val_mse_scores, val_mae_scores = compute_reconstruction_scores(eval_model, val_loader_eval)
test_mse_scores, test_mae_scores = compute_reconstruction_scores(eval_model, test_loader_eval)

# Primary policy uses mean squared reconstruction error scores
threshold = float(np.percentile(val_mse_scores, 100 * (1 - TARGET_FPR)))

y_true = np.concatenate([np.zeros(len(val_mse_scores), dtype=int), np.ones(len(test_mse_scores), dtype=int)])
y_scores = np.concatenate([val_mse_scores, test_mse_scores])
y_pred = (y_scores >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
metrics = {
    # Classification metrics (policy-based)
    'threshold': threshold,
    'target_fpr': TARGET_FPR,
    'realized_fpr': float(fp / max(1, (fp + tn))),
    'accuracy': float((tp + tn) / max(1, len(y_true))),
    'precision': float(tp / max(1, (tp + fp))),
    'recall': float(tp / max(1, (tp + fn))),
    'specificity': float(tn / max(1, (tn + fp))),
    'f1': float((2 * tp) / max(1, (2 * tp + fp + fn))),
    'auroc': float(roc_auc_score(y_true, y_scores)),
    'auprc': float(average_precision_score(y_true, y_scores)),
    'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),

    # Reconstruction summary metrics (MSE/MAE)
    'val_mse_mean': float(np.mean(val_mse_scores)),
    'val_mse_std': float(np.std(val_mse_scores)),
    'val_mse_p95': float(np.percentile(val_mse_scores, 95)),
    'test_mse_mean': float(np.mean(test_mse_scores)),
    'test_mse_std': float(np.std(test_mse_scores)),
    'test_mse_p95': float(np.percentile(test_mse_scores, 95)),
    'val_mae_mean': float(np.mean(val_mae_scores)),
    'val_mae_std': float(np.std(val_mae_scores)),
    'val_mae_p95': float(np.percentile(val_mae_scores, 95)),
    'test_mae_mean': float(np.mean(test_mae_scores)),
    'test_mae_std': float(np.std(test_mae_scores)),
    'test_mae_p95': float(np.percentile(test_mae_scores, 95)),

    # Separation indicators
    'mse_mean_gap_test_minus_val': float(np.mean(test_mse_scores) - np.mean(val_mse_scores)),
    'mae_mean_gap_test_minus_val': float(np.mean(test_mae_scores) - np.mean(val_mae_scores)),

    'val_count': int(len(val_mse_scores)),
    'test_count': int(len(test_mse_scores)),
}

print('\n=== V4 Evaluation (val normal + test anomaly) ===')
print('Classification metrics (FPR@20 policy):')
for k in ['threshold', 'target_fpr', 'realized_fpr', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'auroc', 'auprc']:
    print(f'{k}: {metrics[k]:.6f}')
print(f"Counts -> TP:{metrics['tp']} TN:{metrics['tn']} FP:{metrics['fp']} FN:{metrics['fn']}")

print('\nReconstruction metrics:')
for k in [
    'val_mse_mean', 'val_mse_std', 'val_mse_p95',
    'test_mse_mean', 'test_mse_std', 'test_mse_p95',
    'val_mae_mean', 'val_mae_std', 'val_mae_p95',
    'test_mae_mean', 'test_mae_std', 'test_mae_p95',
    'mse_mean_gap_test_minus_val', 'mae_mean_gap_test_minus_val',
]:
    print(f'{k}: {metrics[k]:.6f}')

# Save compact evaluation artifacts
eval_out = REPO_ROOT / 'results' / 'ecnn_v4_eval'
eval_out.mkdir(parents=True, exist_ok=True)
with open(eval_out / 'v4_eval_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
np.save(eval_out / 'v4_val_mse_scores.npy', val_mse_scores)
np.save(eval_out / 'v4_test_mse_scores.npy', test_mse_scores)
np.save(eval_out / 'v4_val_mae_scores.npy', val_mae_scores)
np.save(eval_out / 'v4_test_mae_scores.npy', test_mae_scores)
print(f'Saved metrics and scores to: {eval_out}')

## Testing And Evaluation (Turbo Zip)

This section evaluates the trained V4 model using the same style as your primary evaluation policy:
- Score: mean reconstruction error (squared)
- Threshold: calibrated on normal validation scores at target FPR 20%
- Evaluation set: normal = val split, anomaly = test split (from test_fast.zip)

## Next Step
Use the new checkpoint in your evaluation notebook by adding a new model config entry (for example: `ecnn_v4`) and run the same pure-equivariance and primary-style robustness evaluations for direct comparison against V3/CNN baselines.